In [ ]:
!pip install -q transformers datasets accelerate sentencepiece openpyxl evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving pershop_intent.xlsx to pershop_intent.xlsx


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# اسم فایل اکسل را اگر فرق دارد، این‌جا عوض کن
df = pd.read_excel("pershop_intent.xlsx")

# مطمئن شو ستون‌ها این نام‌ها را دارند:
print(df.columns)


Index(['intent', 'text'], dtype='object')


In [ ]:
# حذف ردیف‌هایی که خالی‌اند
df = df.dropna(subset=["text", "intent"])

# بر زدن (shuffle)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# ابتدا train/test، بعد dev را از train جدا می‌کنیم
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["intent"])
train_df, dev_df = train_test_split(train_df, test_size=0.125, random_state=42, stratify=train_df["intent"])
# 0.2 تست + 0.8*0.125 ≈ 0.1 dev → حدود 70/10/20

print(len(train_df), len(dev_df), len(test_df))


7736 1106 2211


In [ ]:
from datasets import Dataset, DatasetDict

train_ds = Dataset.from_pandas(train_df[["text", "intent"]])
dev_ds   = Dataset.from_pandas(dev_df[["text", "intent"]])
test_ds  = Dataset.from_pandas(test_df[["text", "intent"]])

dataset = DatasetDict({
    "train": train_ds,
    "validation": dev_ds,
    "test": test_ds
})

dataset


DatasetDict({
    train: Dataset({
        features: ['text', 'intent', '__index_level_0__'],
        num_rows: 7736
    })
    validation: Dataset({
        features: ['text', 'intent', '__index_level_0__'],
        num_rows: 1106
    })
    test: Dataset({
        features: ['text', 'intent', '__index_level_0__'],
        num_rows: 2211
    })
})

In [ ]:
# لیست intentهای یکتا
unique_intents = sorted(df["intent"].unique())
label2id = {label: i for i, label in enumerate(unique_intents)}
id2label = {i: label for label, i in label2id.items()}

label2id, id2label


({'Deny': 0,
  'Goodby': 1,
  'Greeting': 2,
  'Regret': 3,
  'SuperProduct_ask': 4,
  'User_Confirm': 5,
  'ask_detail': 6,
  'chitchat': 7,
  'inform': 8,
  'prev_inform': 9,
  'purchase': 10,
  'user_ask': 11},
 {0: 'Deny',
  1: 'Goodby',
  2: 'Greeting',
  3: 'Regret',
  4: 'SuperProduct_ask',
  5: 'User_Confirm',
  6: 'ask_detail',
  7: 'chitchat',
  8: 'inform',
  9: 'prev_inform',
  10: 'purchase',
  11: 'user_ask'})

In [ ]:
def encode_labels(example):
    example["label"] = label2id[example["intent"]]
    return example

dataset = dataset.map(encode_labels)
dataset = dataset.remove_columns(["intent"])  # فقط text و label بماند
dataset


Map:   0%|          | 0/7736 [00:00<?, ? examples/s]

Map:   0%|          | 0/1106 [00:00<?, ? examples/s]

Map:   0%|          | 0/2211 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', '__index_level_0__', 'label'],
        num_rows: 7736
    })
    validation: Dataset({
        features: ['text', '__index_level_0__', 'label'],
        num_rows: 1106
    })
    test: Dataset({
        features: ['text', '__index_level_0__', 'label'],
        num_rows: 2211
    })
})

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "HooshvareLab/bert-base-parsbert-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

num_labels = len(unique_intents)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at HooshvareLab/bert-base-parsbert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
max_length = 128  # معمولاً برای جملات کوتاه کافی است

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format(type="torch")


Map:   0%|          | 0/7736 [00:00<?, ? examples/s]

Map:   0%|          | 0/1106 [00:00<?, ? examples/s]

Map:   0%|          | 0/2211 [00:00<?, ? examples/s]

In [ ]:
import numpy as np
import evaluate # Changed from `from datasets import load_metric`

accuracy_metric = evaluate.load("accuracy") # Changed from `load_metric("accuracy")`

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    result = accuracy_metric.compute(predictions=preds, references=labels)
    return {"accuracy": result["accuracy"]}

In [ ]:
from transformers import TrainingArguments, Trainer

batch_size = 16
logging_steps = 50

training_args = TrainingArguments(
    output_dir="./pershop_intent_parsbert",
    eval_strategy="epoch", # Changed from evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=logging_steps,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

/tmp/ipython-input-1138706338.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: keyvanmahmoudi to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,0.273200,0.243970,0.933996
2,0.157800,0.246178,0.940325
3,0.089400,0.264426,0.941230


TrainOutput(global_step=1452, training_loss=0.2442222688726814, metrics={'train_runtime': 645.9103, 'train_samples_per_second': 35.931, 'train_steps_per_second': 2.248, 'total_flos': 1526707407790080.0, 'train_loss': 0.2442222688726814, 'epoch': 3.0})

In [ ]:
test_results = trainer.evaluate(tokenized_datasets["test"])
test_results